# How to Speak: QWen & LoRa

Runtime > Change runtime type > T4 GPU, then run all.

The first cell asks how you want the model to speak — defaults to pirate

In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "trl==0.17.0" "bitsandbytes==0.45.5" "datasets==3.6.0" "accelerate==1.7.0"

In [ ]:
import random
import re
import time
from pathlib import Path

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_EXAMPLES = 500
EPOCHS = 3

DEFAULT_STYLE = "a boisterous pirate"
STYLE_OVERRIDE = ""  # set this to answer the question below without being asked

EVAL_PROMPTS = [
    "I used to write really good examples",
    "Explain what a bird is, in two or three sentences.",
    "What is the capital of France?",
    "Give me a hand with my shopping please.",
]

assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

start = time.time()


def log(msg):
    print(f"\n[{time.time() - start:6.1f}s] {msg}\n{'-' * 70}", flush=True)


def ask_style():
    """Whatever you type is the voice. Blank falls back to the pirate."""
    if STYLE_OVERRIDE.strip():
        return STYLE_OVERRIDE.strip()
    try:
        answer = input(
            "How should the model learn to speak?\n"
            "Describe a voice, e.g. 'a Victorian butler' or 'an over-caffeinated surfer'.\n"
            f"Press Enter for the default: {DEFAULT_STYLE}\n> "
        )
    except (EOFError, KeyboardInterrupt, OSError):
        answer = ""  # not an interactive runtime
    return answer.strip() or DEFAULT_STYLE


def slugify(style):
    slug = re.sub(r"[^a-z0-9]+", "-", style.lower()).strip("-")
    if len(slug) > 40:
        slug = slug[:40].rpartition("-")[0]  # don't cut a word in half
    return slug or "style"  # e.g. a voice typed as pure punctuation


STYLE = ask_style()
IS_PIRATE = STYLE == DEFAULT_STYLE
STYLE_SLUG = slugify(STYLE)
ADAPTER_DIR = f"qwen-howtospeak-{STYLE_SLUG}-adapter"

log(f"teaching the model to speak like {STYLE!r}, saving to {ADAPTER_DIR}/")

In [ ]:
# 4-bit NF4 base. T4 doesn't do bfloat16.
use_bf16 = torch.cuda.is_bf16_supported()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

log(
    f"loaded {MODEL_ID} in 4-bit on {torch.cuda.get_device_name(0)}\n"
    f"{model.get_memory_footprint() / 1e9:.2f} GB on device, bf16={use_bf16}"
)

In [ ]:
def generate(m, prompt, max_new_tokens=150):
    """Greedy, so before/after isn't just sampling noise."""
    enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


baseline = {p: generate(model, p) for p in EVAL_PROMPTS}
log(f"baseline sample\n{EVAL_PROMPTS[0]}\n{baseline[EVAL_PROMPTS[0]]}")

In [ ]:
# Dolly answers, restyled into STYLE.
rng = random.Random(0)

# The default pirate gets regex-mangled: instant and deterministic.
WORD_SWAPS = [
    (r"\bhello\b|\bhi\b", "ahoy"),
    (r"\byes\b", "aye"),
    (r"\byou\b", "ye"),
    (r"\byour\b", "yer"),
    (r"\bmy\b", "me"),
    (r"\bis\b|\bare\b|\bam\b", "be"),
    (r"\bfriends\b", "mateys"),
    (r"\bfriend\b", "matey"),
    (r"\bmoney\b", "doubloons"),
    (r"\beveryone\b", "all hands"),
]

OPENERS = ["Arrr! ", "Ahoy, matey! ", "Avast! ", "Shiver me timbers! ", "Yarr! "]
CLOSERS = [" Arrr!", " Yo ho ho!", " Fair winds to ye!", " Now back to swabbin' the deck!"]


def _keep_case(replacement):
    def sub(match):
        return replacement.capitalize() if match.group(0)[0].isupper() else replacement
    return sub


def piratify(text):
    for pattern, replacement in WORD_SWAPS:
        text = re.sub(pattern, _keep_case(replacement), text, flags=re.IGNORECASE)
    if rng.random() < 0.7:
        text = rng.choice(OPENERS) + text
    if rng.random() < 0.4:
        text = text.rstrip() + rng.choice(CLOSERS)
    return text


# Any voice you typed has no regex table, so the base model rewrites the answers
# instead. This cell runs before the adapter exists, so it's plain Qwen talking.
REWRITE_BATCH_SIZE = 16
REWRITE_SYSTEM = (
    f"You rewrite text so that it sounds like {STYLE}. "
    "Keep every fact and all of the meaning exactly as it is. "
    "Change only the voice: word choice, rhythm, and mannerisms. "
    "Reply with the rewritten text and nothing else."
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def restyle_batch(texts):
    """Sampled, not greedy — 500 rewrites off one greedy path all wear the same tic."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": REWRITE_SYSTEM}, {"role": "user", "content": t}],
            add_generation_prompt=True,
            tokenize=False,
        )
        for t in texts
    ]
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # decoder-only
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=320,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    tokenizer.padding_side = padding_side  # SFTTrainer's collator pads the other way
    return [
        tokenizer.decode(row[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().strip('"')
        for row in out
    ]


def restyle_all(texts):
    done = []
    for i in range(0, len(texts), REWRITE_BATCH_SIZE):
        batch = texts[i:i + REWRITE_BATCH_SIZE]
        # A rewrite that collapsed or ran away is a generation failure, not a voice.
        done += [
            new if 0.5 * len(old) < len(new) < 3 * len(old) else old
            for old, new in zip(batch, restyle_batch(batch))
        ]
        print(f"  restyled {len(done)}/{len(texts)}", flush=True)
    return done


raw = load_dataset("databricks/databricks-dolly-15k", split="train")
raw = raw.filter(lambda ex: ex["context"] == "" and 100 < len(ex["response"]) < 600)
raw = raw.shuffle(seed=42).select(range(N_EXAMPLES))

if IS_PIRATE:
    responses = [piratify(r) for r in raw["response"]]
else:
    log(f"rewriting {len(raw)} answers as {STYLE!r} — a few minutes on a T4")
    torch.manual_seed(0)
    responses = restyle_all(raw["response"])

train_ds = Dataset.from_list([
    {
        "messages": [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response},
        ]
    }
    for instruction, response in zip(raw["instruction"], responses)
])

example = train_ds[0]["messages"]
log(f"{len(train_ds)} training pairs, e.g.\nUSER: {example[0]['content']}\nASSISTANT: {example[1]['content']}")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log(f"adapters attached: {trainable / 1e6:.1f}M trainable of {total / 1e6:.0f}M ({100 * trainable / total:.2f}%)")

In [ ]:
# ~10-15 min on a T4. SFTTrainer applies the chat template itself.
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=f"qwen-{STYLE_SLUG}-lora",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        max_seq_length=512,
        logging_steps=5,
        save_strategy="no",
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

metrics = trainer.train().metrics
log(f"trained {trainer.state.global_step} steps in {metrics['train_runtime'] / 60:.1f} min, loss {metrics['train_loss']:.3f}")

In [ ]:
model.eval()
model.config.use_cache = True  # gradient checkpointing turned this off

for p in EVAL_PROMPTS:
    log(f"{p}\n\nBEFORE: {baseline[p]}\n\nAFTER:  {generate(model, p)}")

# Base weights were never touched, so the adapter is a switch.
with model.disable_adapter():
    log(f"adapter disabled\n{generate(model, EVAL_PROMPTS[0])}")

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*") if f.is_file()) / 1e6
log(f"saved to {ADAPTER_DIR}/ ({size_mb:.0f} MB)")